In [ ]:
import pandas as pd
from nrclex import NRCLex
from sklearn.metrics import classification_report, accuracy_score


In [5]:
df_songs = pd.read_csv('dataset.csv')

In [7]:
def get_nrc_emotions(text):
    emotions = NRCLex()
    emotions.load_raw_text(text)
    return emotions.affect_frequencies


In [8]:
emotions = df_songs['lyrics_cleaned'].apply(lambda x: pd.Series(get_nrc_emotions(x)))

df_songs = pd.concat([df_songs,emotions],axis=1)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Define your features (the 10 NRC categories)
features = ['fear', 'anger', 'anticipation', 'trust', 'surprise', 'sadness', 'joy', 'disgust', 'positive', 'negative']
X = df_songs[features]
y = df_songs['parent_genre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# 2. Get the overall accuracy score
accuracy = accuracy_score(y_test, predictions)
print(f"Overall Model Accuracy: {accuracy:.2%}")
print("-" * 30)

# 3. Print the full Summary (Precision, Recall, F1)
print("Classification Report:")
print(classification_report(y_test, predictions))

Overall Model Accuracy: 16.33%
------------------------------
Classification Report:
                  precision    recall  f1-score   support

       Asian Pop       0.08      0.07      0.07        85
       Classical       0.03      0.24      0.05        45
      Electronic       0.43      0.04      0.07       856
    Folk/Country       0.22      0.12      0.16       331
   Hip-Hop & R&B       0.03      0.47      0.06        32
    Jazz & Blues       0.01      0.02      0.01        65
           Latin       0.00      0.00      0.00        16
           Metal       0.28      0.55      0.37       436
             Pop       0.26      0.03      0.06       297
Reggae/Caribbean       0.05      0.08      0.06       100
            Rock       0.25      0.05      0.09       515
       Soul/Funk       0.14      0.17      0.15       155
  World/Regional       0.21      0.57      0.30       154

        accuracy                           0.16      3087
       macro avg       0.15      0.19      

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Define your feature groups
# Assume 'emotions_list' contains the column names you created with NRCLex
emotions_list = ['positive', 'negative']

# X now includes BOTH the text and the numeric emotion columns
X = df_songs[['lyrics_cleaned'] + emotions_list]
y = df_songs['parent_genre']

# 2. Split the data (using stratify to handle imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Create the ColumnTransformer
# This applies different processing to different columns
preprocessor = ColumnTransformer(
    transformers=[
        # Process text column with TF-IDF
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english'), 'lyrics_cleaned'),
        # Process numeric emotion columns with a Scaler (very important for Logistic Regression)
        ('scaler', StandardScaler(), emotions_list)
    ]
)

# 4. Create the final pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced'))
])

# 5. Train the model
print("Training model with combined features...")
model_pipeline.fit(X_train, y_train)

# 6. Make predictions and evaluate
predictions = model_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

Training model with combined features...
Accuracy: 0.37

Classification Report:

                  precision    recall  f1-score   support

       Asian Pop       0.36      0.44      0.39        96
       Classical       0.11      0.41      0.17        44
      Electronic       0.55      0.24      0.34       882
    Folk/Country       0.45      0.54      0.49       322
   Hip-Hop & R&B       0.20      0.55      0.29        29
    Jazz & Blues       0.12      0.32      0.18        57
           Latin       0.07      0.33      0.12        12
           Metal       0.53      0.71      0.60       452
             Pop       0.20      0.22      0.21       285
Reggae/Caribbean       0.42      0.50      0.46        86
            Rock       0.30      0.15      0.20       493
       Soul/Funk       0.16      0.27      0.21       157
  World/Regional       0.64      0.69      0.66       172

        accuracy                           0.37      3087
       macro avg       0.32      0.41      0.33